# 17 - Segmentation metrics: full comparison

Loads the saved test predictions for all six models (three architectures x two arms) and computes the full per-class metric table: Dice, IoU (Jaccard), sensitivity (recall), and precision. No pixel accuracy - microaneurysms are ~0.14% of pixels, so an all-background prediction would score ~99.9% and tell us nothing.

**Attach:** the committed outputs of 16a/16b/16c, both arms (six `_testpreds.npz` files). No GPU.

In [1]:
import os, re, numpy as np, pandas as pd
CLASSES=['Microaneurysms','Haemorrhages','Hard Exudates','Optic Disc']
MODELS=[('unet_scratch','From-scratch U-Net'),
        ('unet_pretrained','Pretrained U-Net'),
        ('segformer','SegFormer')]
ARMS=['unmasked','masked']
THRESH=0.5

In [2]:
# locate every *_testpreds.npz
pred_files={}
for r,_,fs in os.walk('/kaggle/input'):
    for x in fs:
        if x.endswith('_testpreds.npz'):
            pred_files[x]=os.path.join(r,x)
print('Found', len(pred_files), 'prediction files:')
for k in sorted(pred_files): print('  ',k)

Found 6 prediction files:
   segformer_masked_testpreds.npz
   segformer_unmasked_testpreds.npz
   unet_pretrained_masked_testpreds.npz
   unet_pretrained_unmasked_testpreds.npz
   unet_scratch_masked_testpreds.npz
   unet_scratch_unmasked_testpreds.npz


In [3]:
def metrics_for(probs, gts, ci, thr=THRESH):
    p=(probs[:,ci]>thr).astype(np.uint8).reshape(-1)
    g=gts[:,ci].astype(np.uint8).reshape(-1)
    tp=int((p*g).sum()); fp=int((p*(1-g)).sum()); fn=int(((1-p)*g).sum())
    dice=2*tp/(2*tp+fp+fn) if (2*tp+fp+fn)>0 else 0.0
    iou =tp/(tp+fp+fn) if (tp+fp+fn)>0 else 0.0
    sens=tp/(tp+fn) if (tp+fn)>0 else 0.0
    prec=tp/(tp+fp) if (tp+fp)>0 else 0.0
    return dice,iou,sens,prec

rows=[]
for slug,name in MODELS:
    for arm in ARMS:
        fn=f'{slug}_{arm}_testpreds.npz'
        if fn not in pred_files:
            print('MISSING',fn); continue
        d=np.load(pred_files[fn])
        probs=d['probs'].astype(np.float32); gts=d['gts']
        for ci,cls in enumerate(CLASSES):
            dice,iou,sens,prec=metrics_for(probs,gts,ci)
            rows.append(dict(Model=name,Arm=arm,Class=cls,
                             Dice=round(dice,4),IoU=round(iou,4),
                             Sensitivity=round(sens,4),Precision=round(prec,4)))
df=pd.DataFrame(rows)
print('computed', len(df), 'rows')

computed 24 rows


## Full per-class table

In [4]:
pd.set_option('display.width',200,'display.max_rows',60)
print(df.to_string(index=False))
df.to_csv('/kaggle/working/segmentation_metrics_full.csv',index=False)
print('\nSaved segmentation_metrics_full.csv')

             Model      Arm          Class   Dice    IoU  Sensitivity  Precision
From-scratch U-Net unmasked Microaneurysms 0.4679 0.3054       0.4324     0.5097
From-scratch U-Net unmasked   Haemorrhages 0.5309 0.3614       0.4925     0.5758
From-scratch U-Net unmasked  Hard Exudates 0.7662 0.6211       0.7691     0.7634
From-scratch U-Net unmasked     Optic Disc 0.7964 0.6617       0.9264     0.6984
From-scratch U-Net   masked Microaneurysms 0.4473 0.2881       0.4061     0.4979
From-scratch U-Net   masked   Haemorrhages 0.5132 0.3452       0.4911     0.5373
From-scratch U-Net   masked  Hard Exudates 0.7711 0.6274       0.7689     0.7733
From-scratch U-Net   masked     Optic Disc 0.8295 0.7086       0.9408     0.7417
  Pretrained U-Net unmasked Microaneurysms 0.5157 0.3475       0.4943     0.5391
  Pretrained U-Net unmasked   Haemorrhages 0.5675 0.3962       0.5668     0.5682
  Pretrained U-Net unmasked  Hard Exudates 0.8109 0.6819       0.8388     0.7847
  Pretrained U-Net unmasked 

## Unmasked comparison (the core architecture result)

In [5]:
um=df[df.Arm=='unmasked'].pivot(index='Class',columns='Model',values='Dice')
um=um[['From-scratch U-Net','Pretrained U-Net','SegFormer']]
print('DICE by class (unmasked):')
print(um.to_string())
print('\nMean Dice:', um.mean().round(4).to_dict())

DICE by class (unmasked):
Model           From-scratch U-Net  Pretrained U-Net  SegFormer
Class                                                          
Haemorrhages                0.5309            0.5675     0.6541
Hard Exudates               0.7662            0.8109     0.8211
Microaneurysms              0.4679            0.5157     0.5227
Optic Disc                  0.7964            0.9465     0.9478

Mean Dice: {'From-scratch U-Net': 0.6404, 'Pretrained U-Net': 0.7102, 'SegFormer': 0.7364}


## Masked vs unmasked (the masking extension)

In [6]:
piv=df.pivot_table(index=['Model','Class'],columns='Arm',values='Dice')
piv['change']=(piv['masked']-piv['unmasked']).round(4)
print(piv.to_string())
print('\nMean Dice change from masking, by model:')
for slug,name in MODELS:
    sub=df[(df.Model==name)]
    mu=sub[sub.Arm=='unmasked'].Dice.mean(); mm=sub[sub.Arm=='masked'].Dice.mean()
    print(f'  {name:<20} {mm-mu:+.4f}')

Arm                                masked  unmasked  change
Model              Class                                   
From-scratch U-Net Haemorrhages    0.5132    0.5309 -0.0177
                   Hard Exudates   0.7711    0.7662  0.0049
                   Microaneurysms  0.4473    0.4679 -0.0206
                   Optic Disc      0.8295    0.7964  0.0331
Pretrained U-Net   Haemorrhages    0.5799    0.5675  0.0124
                   Hard Exudates   0.8134    0.8109  0.0025
                   Microaneurysms  0.4978    0.5157 -0.0179
                   Optic Disc      0.9484    0.9465  0.0019
SegFormer          Haemorrhages    0.6515    0.6541 -0.0026
                   Hard Exudates   0.8231    0.8211  0.0020
                   Microaneurysms  0.5300    0.5227  0.0073
                   Optic Disc      0.9572    0.9478  0.0094

Mean Dice change from masking, by model:
  From-scratch U-Net   -0.0001
  Pretrained U-Net     -0.0003
  SegFormer            +0.0040


## Notes
- Metrics at threshold 0.5, pooled over all test tiles (micro-averaged per class).
- Dice and IoU measure overlap; sensitivity is recall (did it find the lesion); precision is how many predicted lesion pixels were correct.
- Per-class reporting is essential: mean scores hide that microaneurysms (tiny) are far harder than the optic disc (large). This mirrors the per-class argument made for classification.
- Single seed per model, as in the masking classification study; report as 'for the seed examined'.